# 11.9 Data Skewness

## Learning Objectives

By the end of this lesson you will be able to:

- Understand what Data Skewness is
- Learn why Data Skew causes slow Spark jobs
- Identify symptoms of skewed data
- Detect Data Skew using Spark UI
- Analyze real-world skew examples
- Understand the impact of skew on joins and aggregations

> **Core Rule:** Data Skew is one of the most common causes of poor Spark performance.

## Setup: Initialize Spark & Simulate Skew

Let's start our local Spark Session and intentionally create a highly skewed dataset to demonstrate the problem in PySpark.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import numpy as np

# Initialize a local Spark session
spark = SparkSession.builder \
    .appName("Data_Skewness_Demo") \
    .master("local[*]") \
    .getOrCreate()

# Let's create a dataset with 1,000,000 rows.
# We will force 90% of the rows to belong to customer_id = '999' (The Skew)
# The remaining 10% will be randomly distributed across 100 other customers.

print("Generating Skewed Data...")

# Create the skewed bulk (900,000 rows for customer_id 999)
df_skewed = spark.range(1, 900000).withColumn("customer_id", F.lit(999))

# Create the normal distribution (100,000 rows spread across IDs 1 to 100)
df_normal = spark.range(1, 100000).withColumn("customer_id", (F.rand() * 100).cast("int"))

# Combine them
df = df_skewed.union(df_normal)
print("Mock Skewed Data Created!")

# Why Learn Data Skew?

Many Spark jobs appear to have:
- Enough executors
- Enough memory
- Proper cluster configuration

Yet they still run slowly. A common reason is **Data Skew**.

Data Skew causes some tasks to do significantly more work than others. As a result, the entire job becomes slow.

# Real-World Scenario

Imagine a classroom with 100 students. The teacher assigns homework grading.

**Scenario 1 (Balanced):**
- 10 teachers
- Each teacher grades 10 papers.
- Workload is balanced. Everyone finishes at the same time.

**Scenario 2 (Skewed):**
- One teacher grades 90 papers.
- Nine teachers grade 1 paper each.
- Workload is highly unbalanced. The class cannot go home until the first teacher finishes grading all 90 papers.

This imbalance is exactly how Data Skew affects Spark executors.

# What is Data Skew?

Data Skew occurs when data is not distributed evenly across partitions.

Some partitions contain very large amounts of data, while others contain very little data.

This creates massively uneven workloads across executors.

<h3>Balanced vs Skewed Data Distribution</h3>

<img src="../assets/Module_11/11_09_01.png" alt="Balanced vs Skewed Partitions" width="75%">

<p><i><b>Image Prompt:</b> Side-by-side comparison of balanced Spark partitions versus skewed partitions. Balanced partitions contain similar amounts of data, while one skewed partition contains most records. Educational infographic.</i></p>

# Balanced Partition Example

- Partition 1: 250 MB
- Partition 2: 240 MB
- Partition 3: 260 MB
- Partition 4: 250 MB

All executors receive similar workloads. Processing remains efficient.

# Skewed Partition Example

- Partition 1: 50 MB
- Partition 2: 40 MB
- Partition 3: 60 MB
- **Partition 4: 5 GB**

One partition contains most of the data. One executor now performs significantly more work, and might even run out of memory.

<h3>Skewed Partition Example</h3>

<img src="../assets/Module_11/11_09_02.png" alt="Skewed Partition Warning" width="75%">

<p><i><b>Image Prompt:</b> Spark cluster showing one partition containing several gigabytes of data while other partitions contain very little. Visualization of severe data skew.</i></p>

# Why Is Data Skew a Problem?

Spark processes partitions in parallel. Ideally, all tasks finish at approximately the same time.

With skew:
- 199 tasks finish in 5 seconds.
- 1 task continues running for 20 minutes.

Because Spark operates in stages, the entire stage must wait for that single task to complete before moving to the next stage.

# The Slowest Task Determines Completion Time

Suppose:
- Task 1: 10 Seconds
- Task 2: 12 Seconds
- Task 3: 11 Seconds
- **Task 4: 5 Minutes**

The stage cannot finish until Task 4 finishes. This is called the **"straggler problem."**

<h3>Straggler Task Problem</h3>

<img src="../assets/Module_11/11_09_03.png" alt="Straggler Task" width="75%">

<p><i><b>Image Prompt:</b> Spark stage visualization showing multiple tasks finishing quickly while one task continues running much longer. Educational illustration of straggler tasks caused by skew.</i></p>

# Common Causes of Data Skew

Data Skew often occurs because:
- Certain keys appear very frequently (e.g., NULL values or default values like `"Unknown"`)
- Customer activity is uneven (e.g., Power users)
- Product popularity is uneven (e.g., A viral product)
- Some categories dominate the dataset

Real-world data is rarely perfectly balanced.

# Example: E-Commerce Orders

Suppose we group orders by product.

Most products sell: 100–500 units.
But one viral product sells: 5 Million units.

If you run a `groupBy("product_id")`, this popular product creates a highly skewed partition.

<h3>Popular Product Skew</h3>

<img src="../assets/Module_11/11_09_04.png" alt="Popular Product Distribution" width="75%">

<p><i><b>Image Prompt:</b> Product sales distribution chart showing one extremely popular product generating millions of records while most products have relatively few records. Data skew visualization.</i></p>

# Other Common Examples

**Banking Transactions:**
- Most accounts: 10–50 transactions
- One corporate account: 10 Million transactions

**Website Traffic:**
- Most users: 1–20 visits
- One automated bot: 50 Million visits

Aggregating by these IDs will create massive bottlenecks.

<h3>Real-World Skew Examples</h3>

<img src="../assets/Module_11/11_09_05.png" alt="Real World Skew Examples" width="75%">

<p><i><b>Image Prompt:</b> Multiple examples of skewed datasets including e-commerce products, banking accounts and website traffic. Educational infographic demonstrating uneven data distribution.</i></p>

# Data Skew During Joins and Aggregations

Data Skew becomes especially dangerous during Wide Transformations (operations that cause a Shuffle).

This includes:
- `join()`
- `groupBy()`
- `count()`
- `sum()`

During shuffle, all identical keys are sent to the exact same partition. If one key has 10 million rows, one executor must process 10 million rows.

<h3>Skew During Join and Aggregation</h3>

<img src="../assets/Module_11/11_09_06.png" alt="Skew during Shuffle" width="75%">

<p><i><b>Image Prompt:</b> Spark shuffle operation showing a heavily skewed key causing one partition to receive significantly more records than others during join and aggregation.</i></p>

# Detecting Data Skew

Data Skew can often be identified through:
- Spark UI (Stages Tab)
- Task Duration anomalies
- High Shuffle Read/Write values on a single executor
- Programmatic Key Frequency Analysis

Let's look at how to detect it using PySpark code.

# Examining Key Distribution (Programmatic Detection)

A useful technique before performing a massive Join is to check the key frequency.

Count the occurrences of each key. Extremely frequent keys guarantee skew.

In [ ]:
print("Analyzing Customer ID Frequency to Detect Skew...")
print("-"*50)

# Group by the key and order descending to see the biggest culprits
skew_analysis = df.groupBy("customer_id") \
                  .count() \
                  .orderBy("count", ascending=False)

skew_analysis.show(5)

print("WARNING! The results show Customer ID 999 has almost 900,000 records.")
print("The next highest customer only has ~1,000 records.")
print("If we join on this column, we will suffer from severe Data Skew.")

<h3>Key Frequency Analysis</h3>

<img src="../assets/Module_11/11_09_08.png" alt="Key Frequency Chart" width="75%">

<p><i><b>Image Prompt:</b> Frequency distribution chart showing one dominant key with extremely high occurrence count compared to all other keys. Data skew detection infographic.</i></p>

# Detecting Skew in Spark UI

If you don't catch it programmatically, you will catch it in the Spark UI.

Open the **Stages Tab** and look for:
- One task taking much longer than the 75th percentile.
- Uneven task durations.
- High shuffle read/write values isolated to a single executor.

<h3>Detecting Skew in Spark UI</h3>
<img src="../assets/Module_11/11_09_07.png" alt="Spark UI Skew" width="75%">
<p><i><b>Image Prompt:</b> Spark UI Stages Tab showing one task significantly longer than all others. Highlighted metrics indicating possible data skew. Educational dashboard visualization.</i></p>

# Business Impact

Data Skew can cause:
- Extremely slow pipelines
- Delayed business reporting
- Increased cloud costs (executors sit idle waiting for the straggler)
- Job failures (`OutOfMemoryError`)

Identifying skew early is critical for production systems.

# Real Production Example

**Daily Sales Job:**
- Expected Runtime: 10 Minutes
- Actual Runtime: 45 Minutes

**Investigation:**
- Cluster healthy? Yes.
- Memory sufficient? Yes.
- CPU utilization normal? Yes.

**Root Cause:** One highly skewed customer key (a bot or a massive corporate account). This single key forced one executor to process 90% of the data, slowing the entire pipeline.

<h3>Production Impact of Data Skew</h3>

<img src="../assets/Module_11/11_09_09.png" alt="Production Impact" width="75%">

<p><i><b>Image Prompt:</b> Production Spark pipeline slowed by a skewed partition. Timeline comparison showing expected runtime versus actual runtime due to data skew.</i></p>

# Key Takeaways

- Data Skew occurs when partitions are unevenly distributed during a shuffle.
- It causes some tasks (stragglers) to receive significantly more work than others.
- Data Skew commonly ruins the performance of Joins and Aggregations.
- Spark UI helps detect skew visually.
- Key frequency analysis (`groupBy().count()`) can reveal skewed values programmatically.

Data Skew is one of the most common Spark performance challenges.

---

# Next Lesson: 11.10 Fixing Data Skew

Understanding skew is the first step. Fixing skew is the next.

In the next lesson we will learn:
- Salting techniques
- Broadcast-based solutions
- Adaptive Query Execution (AQE) skew handling